离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch02_python_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第2章：用Python处理收益数据

读取构造价格夹具 `data/fixtures/a1_prices.csv`，证券标识是虚构的永久ID。先核Python列表与NumPy复利，再以pandas调用共用收益函数，并用Polars读取同一记录。时间均为UTC；收益为相邻收盘价变化，不填补缺失值。月末重采样的period_end是分组标签，observation_time才是最后实际输入行时间。相关系数先按共同时间成对删缺失；末格新画构造收益图。

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from quant_course import compute_returns

fixture = ROOT / "data" / "fixtures" / "a1_prices.csv"
long = pd.read_csv(fixture)
long["timestamp"] = pd.to_datetime(long["timestamp"], utc=True)
example_prices = [100.0, 110.0, 99.0]
price_array = np.asarray(example_prices, dtype=float)
hand_returns = price_array[1:] / price_array[:-1] - 1.0
hand_cumulative_return = float(np.prod(1.0 + hand_returns) - 1.0)
print("constructed Python list:", example_prices)
print("NumPy vectorized returns:", hand_returns)
print(f"cumulative return: {hand_cumulative_return:.2%}")


In [ ]:
panel = long.pivot(index="timestamp", columns="security_id", values="close").sort_index()
returns = panel.apply(compute_returns)
rolling_mean = returns.rolling(window=2, min_periods=2).mean()
expanding_vol = returns.expanding(min_periods=2).std(ddof=1)
lagged_returns = returns.shift(1)
cross_section_rank = returns.rank(axis=1, method="average", pct=True)
security_mean = long.groupby("security_id", sort=True)["close"].mean()
monthly_input = long.assign(observation_time=long["timestamp"]).set_index("timestamp")
monthly_sample = (
    monthly_input.groupby("security_id")
    .resample("ME")[["observation_time", "close"]]
    .last(skipna=False)
)
monthly_sample.index = monthly_sample.index.set_names(["security_id", "period_end"])
print("simple returns:", returns)
print("rolling 2-row means:", rolling_mean)
print("expanding sample volatility:", expanding_vol)
print("shift(1), which puts the prior row at this row:", lagged_returns)
print("same-date percentile rank:", cross_section_rank)
print("groupby mean close:", security_mean)
print("resample bins: period_end is a bin label; observation_time is the last actual input row:")
print(monthly_sample)


In [ ]:
pl_long = pl.read_csv(fixture).with_columns(
    pl.col("timestamp").str.to_datetime(time_zone="UTC", strict=True)
).sort(["security_id", "timestamp"])
pl_returns = pl_long.with_columns(
    (pl.col("close") / pl.col("close").shift(1).over("security_id") - 1.0).alias("simple_return")
).with_columns(
    pl.col("simple_return").rank(method="average").over("timestamp").alias("date_rank")
)
pl_group_means = pl_long.group_by("security_id").agg(pl.col("close").mean().alias("mean_close"))
print(pl_group_means)
pl_returns.select("security_id", "timestamp", "simple_return", "date_rank")


In [ ]:
# Pair rows on their timestamp first; separate dropna calls can pair different dates.
paired_returns = returns[["SIM-ALPHA", "SIM-BETA"]].dropna()
rho = spearmanr(paired_returns["SIM-ALPHA"], paired_returns["SIM-BETA"]).statistic
print(f"SciPy Spearman rho on pairwise-aligned returns: {rho:.4f}")
ax = returns.plot(marker="o", figsize=(8, 4), title="Simulated daily simple returns")
ax.set_xlabel("UTC close timestamp")
ax.set_ylabel("Simple return")
ax.axhline(0, color="#333333", linewidth=0.8)
ax.grid(alpha=0.25)
fig = ax.get_figure()
fig.tight_layout()
output = RUN_DIR / "figures" / "a1" / "ch02_returns.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=160)
plt.close(fig)
output


## 可选历史缓存示例（默认不执行）

下面保留的围栏代码说明原French十行业月收益缓存的描述用途。市场文件不随本语言配套提供，当前既无再分发/使用授权，也未授权执行该分支。它不替换上面的构造数据，不估计策略或Sharpe。只有合法取得且符合原字节契约的输入才能谈历史数字复现。

历史French分支默认跳过：G01权利unknown，G05/A10-B未授权，无市场文件随包。以下围栏仅保留历史源说明，不执行；其中QA路径不是便携包运行依赖。

```python
french_path = ROOT / "data" / "processed" / "french_10_industries_monthly.csv"
manifest_path = ROOT / "data" / "manifests" / "french_industries_2026-10-05.json"
if french_path.exists():
    french = pd.read_csv(french_path)
    french["month"] = pd.to_datetime(french["month"].astype(str), format="%Y%m")
    industry_columns = [column for column in french.columns if column != "month"]
    assert len(french) == 1202
    assert french.loc[0, "NoDur"] == 0.0144
    ax = french.set_index("month")[industry_columns].plot(
        figsize=(10, 5), linewidth=0.7, alpha=0.8,
        title="French 10 Industry monthly returns (source returns)"
    )
    ax.set_xlabel("Month")
    ax.set_ylabel("Monthly return")
    ax.grid(alpha=0.2)
    fig = ax.get_figure()
    fig.tight_layout()
    output = RUN_DIR / "figures" / "a1" / "ch02_french_monthly_returns.png"
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=160)
    plt.close(fig)
    print(f"Source manifest: {manifest_path}")
    print(f"Rows: {len(french)}; first month: {french['month'].iloc[0]:%Y-%m}; last month: {french['month'].iloc[-1]:%Y-%m}")
    print(f"Observed-data figure: {output}")
else:
    print(f"French cache absent; no observed-data figure generated: {french_path}")

```